# ⚡ Kronumos 14B Kairos — Scaled Fine-Tuning Pipeline (5K Train / 1K Val)
**Base Architecture:** `Qwen/Qwen2.5-Coder-14B-Instruct` (Apache-2.0 License)
**Compute Tier:** Google Colab Pro / NVIDIA A100 (40GB/80GB SXM4)
**Dataset:** Scaled to 5,000 Training pairs + 1,000 Validation pairs with Zero Train-Test Leakage
**Anti-Overfitting:** LoRA Dropout (0.05) + Weight Decay (0.05) + 2 Epochs + Early Stopping
**Output:** 16-bit Merged Model + GGUF Q4_K_M for Ollama Hub (`kronumos/Kronumos-14B-kairos`)

### Step 1: Verify Hardware & GPU Environment
Verify NVIDIA A100 accelerator and bfloat16 support.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "❌ CUDA GPU required! Switch runtime to A100 in Runtime -> Change runtime type."
device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ Active GPU: {device_name} ({vram_gb:.1f} GB VRAM)")
print(f"✅ bfloat16 supported: {torch.cuda.is_bf16_supported()}")

# Install Unsloth & accelerated stack (fast pre-built wheels, 0 slow C++ compiling)
!pip install unsloth
!pip install --upgrade --no-deps "trl<0.9.0" peft accelerate bitsandbytes datasets
print("✅ Unsloth stack & datasets installed successfully!")

In [ ]:
# Install Unsloth, Hugging Face acceleration stack and datasets
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets
print("✅ Unsloth stack & datasets installed successfully!")

### Step 3: Clone Repository & Sync Existing Curated Samples

In [ ]:
import os
if not os.path.exists("Kronomus"):
    !git clone https://github.com/Tokenectomy-Labs/Kronomus.git
    %cd Kronomus
else:
    %cd Kronomus
    !git pull

print(f"📂 Working directory: {os.getcwd()}")
!ls -lh dataset/

### Step 4: Scale Dataset to 5,000 Training & 1,000 Validation Pairs
Pulls surgical, single-file bug fixes from `princeton-nlp/SWE-bench` train split (19,008 real PRs)
and combines with existing curated samples. Strictly decontaminates against all 500 SWE-bench Verified test instances.

In [ ]:
import json, re, random
from datasets import load_dataset, Dataset

KRONUMOS_SYSTEM_PROMPT = (
    "You are Kronumos Kairos v2, an autonomous bug-remediation engine natively integrated with "
    "the Tokenectomy M2M Sub-Cortex. You synthesize surgical, production-safe code fixes with zero dirty diffs.\n\n"
    "OPERATIONAL PROTOCOL:\n"
    "1. Always wrap your diagnostic analysis inside <thought>...</thought> tags before emitting code. "
    "Formulate: (a) Fault hypothesis from the Cleaned Technical Specification, (b) Verified repository target file and symbols, "
    "(c) Minimal defensive patch preserving full backward compatibility.\n"
    "2. NEVER modify code blocks tagged as [USER_REPRODUCTION_SNIPPET - REFERENCE ONLY, NEVER PATCH THIS]. "
    "Target ONLY real internal source files within the repository package tree.\n"
    "3. To apply an atomic change, emit a single SEARCH/REPLACE block with character-exact indentation:\n"
    "   File: path/to/internal/file.py\n"
    "   <<<<<<< SEARCH\n"
    "   original exact code lines\n"
    "   =======\n"
    "   replacement code lines\n"
    "   >>>>>>> REPLACE\n"
    "4. Invariants: NEVER return None from constructors (__new__, __init__). NEVER introduce naked pass in exception handlers. "
    "Guarantee 100% syntactic and structural AST compliance."
)

# 1. Load protected 500 SWE-bench Verified instance IDs (Strict 0-leakage)
print("🔒 Enforcing SWE-bench Verified decontamination gate...")
sb_verified = load_dataset("princeton-nlp/SWE-bench_Verified", split="test")
protected_ids = {x["instance_id"] for x in sb_verified}
print(f"   Guarding {len(protected_ids)} protected test instance IDs.")

# 2. Load existing 1,522 curated pairs
all_train = []
with open("dataset/kairos_v2_train.jsonl", "r", encoding="utf-8") as f:
    all_train = [json.loads(line) for line in f if line.strip()]

all_val = []
with open("dataset/kairos_v2_val.jsonl", "r", encoding="utf-8") as f:
    all_val = [json.loads(line) for line in f if line.strip()]

print(f"📦 Seed curated samples: {len(all_train)} train, {len(all_val)} val.")

# 3. Stream & filter surgical fixes from SWE-bench train split (19,008 candidates)
print("📥 Loading princeton-nlp/SWE-bench (train split)...")
swe_train = load_dataset("princeton-nlp/SWE-bench", split="train")

def parse_unified_diff(diff_text):
    files = re.findall(r"--- a/([^\s\n]+)", diff_text)
    if not files:
        files = re.findall(r"diff --git a/([^\s\n]+)", diff_text)
    if len(set(files)) != 1:
        return None
    fpath = files[0].strip()
    if any(fpath.endswith(ext) for ext in [".rst", ".md", ".txt", ".html", ".css", ".svg", ".png"]):
        return None
    if fpath.startswith("tests/") or "/tests/" in fpath or fpath.startswith("testing/"):
        return None
    orig_lines, new_lines = [], []
    in_hunk = False
    for line in diff_text.splitlines():
        if line.startswith("@@"):
            in_hunk = True
            continue
        if not in_hunk:
            continue
        if line.startswith("-") and not line.startswith("---"):
            orig_lines.append(line[1:])
        elif line.startswith("+") and not line.startswith("+++"):
            new_lines.append(line[1:])
        elif line.startswith(" "):
            orig_lines.append(line[1:])
            new_lines.append(line[1:])
    if not orig_lines or not new_lines or len(orig_lines) > 50 or len(new_lines) > 50:
        return None
    return fpath, "\n".join(orig_lines), "\n".join(new_lines)

seen_ids = {s.get("instance_id", "") for s in all_train + all_val}
new_candidates = []

for item in swe_train:
    iid = item.get("instance_id", "")
    if iid in protected_ids or iid in seen_ids:
        continue
    prob = item.get("problem_statement", "")
    patch = item.get("patch", "")
    if not prob or not patch or len(prob) < 60:
        continue
    parsed = parse_unified_diff(patch)
    if not parsed:
        continue
    fpath, orig, new = parsed
    repo = item.get("repo", "")
    summary = prob.strip().splitlines()[0][:100]
    thought = (
        f"[Step 1: Anomaly & Target Symbol Diagnosis]\n"
        f"Target Component: `{fpath}`\n"
        f"Defect Vector: {summary}\n\n"
        f"[Step 2: Invariant Check & Defense]\n"
        f"Enforcing strict AST integrity, defensive boundary guards, and backward compatibility.\n\n"
        f"[Step 3: Surgical Mutation Execution]\n"
        f"Synthesizing atomic replacement block on `{fpath}`."
    )
    user_msg = f"Repository: {repo}\nIssue ID: {iid}\n\nProblem Description:\n{prob.strip()}"
    asst_msg = f"<thought>\n{thought}\n</thought>\n\nFile: {fpath}\n<<<<<<< SEARCH\n{orig.strip()}\n=======\n{new.strip()}\n>>>>>>> REPLACE"
    new_candidates.append({
        "instance_id": iid,
        "repo": repo,
        "messages": [
            {"role": "system", "content": KRONUMOS_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": asst_msg}
        ]
    })
    seen_ids.add(iid)

print(f"✨ Successfully extracted {len(new_candidates)} clean, decontaminated candidates.")
random.seed(42)
random.shuffle(new_candidates)

# Scale to exactly 5,000 Train and 1,000 Val
TARGET_TRAIN = 5000
TARGET_VAL = 1000
needed_train = max(0, TARGET_TRAIN - len(all_train))
needed_val = max(0, TARGET_VAL - len(all_val))

final_train = all_train + new_candidates[:needed_train]
final_val = all_val + new_candidates[needed_train:needed_train + needed_val]
random.shuffle(final_train)
random.shuffle(final_val)

print(f"🎯 Final Scaled Training Set:   {len(final_train)} samples")
print(f"🎯 Final Scaled Validation Set: {len(final_val)} samples")

train_ds = Dataset.from_list(final_train)
val_ds = Dataset.from_list(final_val)

### Step 5: Initialize FastLanguageModel (Qwen2.5-Coder-14B-Instruct)
Loads Qwen 14B in 4-bit NF4 with 8,192 context window. Configures LoRA with `lora_dropout=0.05` to prevent memorization and overfitting.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

MAX_SEQ_LENGTH = 8192
BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-14B-Instruct"

print(f"🚀 Loading FastLanguageModel: {BASE_MODEL_NAME} (4-bit NF4)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16 if is_bfloat16_supported() else torch.float16,
)

print("⚡ Configuring Regularized LoRA Adapters (r=16, alpha=32, dropout=0.05)...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,  # Anti-overfitting regularization
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

### Step 6: ChatML Token Formatting & Response-Only Loss Masking
Applies the ChatML template and masks user and system prompts from loss computation.

In [ ]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

def formatting_prompts_func(batch):
    convos = batch["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

train_ds = train_ds.map(formatting_prompts_func, batched=True)
val_ds = val_ds.map(formatting_prompts_func, batched=True)
print("✅ ChatML tokens formatted with response-only boundary markers!")

### Step 7: Execute Fine-Tuning with Early Stopping & Validation Tracking
Trained on 5,000 samples across 2 epochs (~625 optimizer steps). Monitored every 50 steps against 1,000 validation samples.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=2,  # 2 epochs optimal for 5,000 samples
        learning_rate=1.5e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="paged_adamw_8bit",
        weight_decay=0.05,  # Anti-overfitting regularization
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./kronumos_14b_checkpoints",
        save_total_limit=2,
        report_to="none"
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Enforce response-only loss masking
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("⚡ Starting Scaled Kronumos 14B Fine-Tuning (5,000 samples, 2 epochs) on A100...")
trainer_stats = trainer.train()
print(f"✅ Training finished in {trainer_stats.metrics['train_runtime']/60:.2f} minutes!")

### Step 8: Export Merged 16-bit Model & GGUF (Q4_K_M) for Ollama
Merges best LoRA weights into base model and outputs `kronumos-14b-kairos-q4_k_m.gguf`.

In [ ]:
OUTPUT_LORA_DIR = "./kronumos_14b_kairos_lora"
OUTPUT_MERGED_DIR = "./kronumos_14b_kairos_merged_16bit"
OUTPUT_GGUF_DIR = "./kronumos_14b_gguf"

# 1. Save LoRA adapter
model.save_pretrained(OUTPUT_LORA_DIR)
tokenizer.save_pretrained(OUTPUT_LORA_DIR)
print(f"💾 Best LoRA adapter saved to {OUTPUT_LORA_DIR}")

# 2. Save 16-bit merged model
print("🔄 Merging LoRA into 16-bit weights...")
model.save_pretrained_merged(OUTPUT_MERGED_DIR, tokenizer, save_method="merged_16bit")
print(f"💾 Merged 16-bit model saved to {OUTPUT_MERGED_DIR}")

# 3. Export to GGUF format (Q4_K_M)
print("📦 Exporting to GGUF Q4_K_M for Ollama Hub...")
model.save_pretrained_gguf(OUTPUT_GGUF_DIR, tokenizer, quantization_method="q4_k_m")
print(f"✅ GGUF file exported to {OUTPUT_GGUF_DIR}!")
!ls -lh {OUTPUT_GGUF_DIR}

### Step 9: Backup GGUF Artifacts to Google Drive

In [ ]:
from google.colab import drive
import shutil, glob

drive.mount('/content/drive')
drive_dest = "/content/drive/MyDrive/kronumos_14b_release"
os.makedirs(drive_dest, exist_ok=True)

print(f"📁 Copying GGUF binary to Google Drive ({drive_dest})...")
for f in glob.glob("kronumos_14b_gguf/*.gguf"):
    shutil.copy2(f, drive_dest)
    print(f"✅ Copied {os.path.basename(f)} to Google Drive!")

print("🎉 Kronumos 14B Kairos scaled fine-tuning & GGUF export complete!")